# Lesson 6 — Database Agent (Natural Language → SQL)

## What you will learn
- Build a **real SQLite database** programmatically
- Create tools that **inspect schema** and **execute SQL**
- Let an LLM write SQL from natural language questions
- **Safe execution** — read-only guard on all queries

## Agent Strategy
```
Question: "Who are the top 3 highest paid employees?"
   ↓
[agent] → list_tables()           → "departments, employees, sales, products"
        → describe_table('employees') → column names + sample rows
        → run_sql('SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 3')
        → "The top 3 highest paid are: Alice ($95k), Hank ($90k), Bob ($85k)"
```

## Why inspect schema first?
LLMs **hallucinate column names**. By always calling `describe_table()` first,
the agent knows the exact column names before writing SQL.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
from config import get_ollama_model

import sqlite3
import os
from typing import Annotated
from typing_extensions import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

## Step 1 — Build the Sample Database

In [ ]:
DB_PATH = "company.db"

conn = sqlite3.connect(DB_PATH)
conn.executescript("""
    DROP TABLE IF EXISTS employees; DROP TABLE IF EXISTS departments;
    DROP TABLE IF EXISTS sales; DROP TABLE IF EXISTS products;

    CREATE TABLE departments (id INTEGER PRIMARY KEY, name TEXT, location TEXT, budget REAL);
    CREATE TABLE employees  (id INTEGER PRIMARY KEY, name TEXT, department_id INTEGER, salary REAL, hire_date TEXT, job_title TEXT);
    CREATE TABLE products   (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL, stock INTEGER);
    CREATE TABLE sales      (id INTEGER PRIMARY KEY, employee_id INTEGER, product_id INTEGER, quantity INTEGER, sale_date TEXT, total REAL);

    INSERT INTO departments VALUES (1,'Engineering','New York',500000),(2,'Sales','Chicago',300000),(3,'Marketing','Los Angeles',200000);
    INSERT INTO employees   VALUES (1,'Alice Johnson',1,95000,'2020-03-15','Senior Engineer'),
                                   (2,'Bob Smith',1,85000,'2021-07-01','Engineer'),
                                   (3,'Carol White',2,72000,'2019-11-20','Sales Manager'),
                                   (4,'David Brown',2,65000,'2022-01-10','Sales Rep'),
                                   (5,'Eve Davis',2,67000,'2021-05-15','Sales Rep'),
                                   (6,'Frank Miller',3,78000,'2020-08-30','Marketing Lead'),
                                   (7,'Grace Wilson',3,70000,'2018-04-12','HR Manager'),
                                   (8,'Hank Taylor',1,90000,'2019-09-05','Lead Engineer');
    INSERT INTO products    VALUES (1,'Laptop Pro','Electronics',1299.99,50),
                                   (2,'Wireless Mouse','Electronics',29.99,200),
                                   (3,'Standing Desk','Furniture',499.99,30),
                                   (4,'Monitor 4K','Electronics',599.99,75);
    INSERT INTO sales       VALUES (1,3,1,5,'2024-01-15',6499.95),(2,4,2,20,'2024-01-20',599.80),
                                   (3,5,4,8,'2024-02-05',4799.92),(4,3,3,3,'2024-02-10',1499.97),
                                   (5,4,1,3,'2024-03-05',3899.97),(6,5,1,7,'2024-03-15',9099.93);
""")
conn.commit()
conn.close()
print(f"✅ Database created: {DB_PATH}")

## Step 2 — Database Tools

We give the agent 4 tools. Note how each docstring tells the LLM **when** to use it.

In [ ]:
@tool
def list_tables() -> str:
    """List all tables in the database. ALWAYS call this first."""
    conn = sqlite3.connect(DB_PATH)
    tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
    conn.close()
    return "Tables: " + ", ".join(tables)


@tool
def describe_table(table_name: str) -> str:
    """Get schema and sample data for a table. Call BEFORE writing SQL."""
    conn = sqlite3.connect(DB_PATH)
    cols   = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
    sample = conn.execute(f"SELECT * FROM {table_name} LIMIT 2").fetchall()
    conn.close()
    col_names = ", ".join(c[1] for c in cols)
    return f"Table '{table_name}':\n  Columns: {col_names}\n  Sample: {sample}"


@tool
def get_table_relationships() -> str:
    """Get foreign key relationships to know how to JOIN tables."""
    conn = sqlite3.connect(DB_PATH)
    rels = []
    for tbl in [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]:
        for fk in conn.execute(f"PRAGMA foreign_key_list({tbl})").fetchall():
            rels.append(f"  {tbl}.{fk[3]} → {fk[2]}.{fk[4]}")
    conn.close()
    return "Relationships:\n" + "\n".join(rels) if rels else "No FK relationships."


@tool
def run_sql(query: str) -> str:
    """
    Execute a SQL SELECT query. Returns up to 20 rows.
    READ-ONLY: only SELECT is allowed.
    Use JOINs when data spans multiple tables.
    """
    if not query.strip().upper().startswith("SELECT"):
        return "ERROR: Only SELECT is allowed."
    conn = sqlite3.connect(DB_PATH)
    try:
        cur = conn.execute(query)
        rows = cur.fetchmany(20)
        cols = [d[0] for d in cur.description]
        if not rows: return "No results."
        out = f"Columns: {', '.join(cols)}\n"
        out += "\n".join("  " + " | ".join(str(v) for v in r) for r in rows)
        return out
    except sqlite3.Error as e:
        return f"SQL Error: {e}"
    finally:
        conn.close()


db_tools = [list_tables, describe_table, get_table_relationships, run_sql]
print(f"✅ {len(db_tools)} database tools ready")

## Step 3 — Build the Database Agent

In [ ]:
class DBState(TypedDict):
    messages: Annotated[list, add_messages]


SYSTEM = """You are an expert SQL database analyst.

ALWAYS follow this order:
1. list_tables() — see what's available
2. describe_table() — check column names (NEVER guess them)
3. get_table_relationships() if you need JOINs
4. run_sql() — execute your SELECT query
5. Give a clear natural-language answer with the data

If SQL fails, re-read the schema and fix it."""


llm = ChatOllama(model=get_ollama_model(), temperature=0)
llm_with_tools = llm.bind_tools(db_tools)


def agent_node(state: DBState) -> dict:
    msgs = [SystemMessage(content=SYSTEM)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(msgs)]}


def should_continue(state: DBState) -> str:
    last = state["messages"][-1]
    return "tools" if (hasattr(last, "tool_calls") and last.tool_calls) else "end"


builder = StateGraph(DBState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(db_tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
builder.add_edge("tools", "agent")
graph = builder.compile()

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except:
    print("Graph compiled!")

## Step 4 — Ask Questions in Natural Language

In [ ]:
def ask(question):
    print(f"\n❓ {question}")
    r = graph.invoke({"messages": [HumanMessage(content=question)]})
    print(f"💬 {r['messages'][-1].content}")

ask("How many employees are in each department?")

In [ ]:
ask("Who are the top 3 highest-paid employees?")

In [ ]:
ask("What is the total sales revenue by employee? Show employee names.")

In [ ]:
ask("What is the average salary per department?")

## Key Takeaways

| Concept | Why it matters |
|---------|----------------|
| `list_tables()` first | LLM must know what exists |
| `describe_table()` before SQL | Prevents hallucinated column names |
| Read-only guard | Safety — agents can't accidentally delete data |
| JOINs | Needed when data spans multiple tables |
| Docstrings | The LLM reads these to know when to use each tool |

## 🏋️ Exercise
1. Add a `projects` table (id, name, department_id, budget, status)
2. Insert 4 projects across different departments
3. Ask: `"Which department has the highest total project budget?"`
4. Ask: `"List all active projects with their department names"`

In [ ]:
# Your exercise here
